# 01 · Data preparation

Builds the job corpus and the resume evaluation sets used by notebooks 02 and 03.

**Data:** [LinkedIn Job Postings (2023-24)](https://www.kaggle.com/datasets/arshkon/linkedin-job-postings) by arshkon (CC BY-SA 4.0) and the [Resume Dataset](https://www.kaggle.com/datasets/snehaanbhawal/resume-dataset) by snehaanbhawal. Set `BASE_DIR` to where the data should be stored.

## Download

In [ ]:
import kagglehub
import shutil
import os

BASE_DIR = r"C:\Users\LENOVO\Documents\Datasets"
JOBS_DIR = os.path.join(BASE_DIR, "linkedin-job-postings")
RESUMES_DIR = os.path.join(BASE_DIR, "resume-dataset")

os.makedirs(BASE_DIR, exist_ok=True)

# Download to kagglehub cache, then move into the Datasets folder (no copy left behind)
jobs_cache = kagglehub.dataset_download("arshkon/linkedin-job-postings")
resumes_cache = kagglehub.dataset_download("snehaanbhawal/resume-dataset")

for src, dst in [(jobs_cache, JOBS_DIR), (resumes_cache, RESUMES_DIR)]:
    if os.path.exists(dst):
        print(f"Already exists, skipping move: {dst}")
    else:
        shutil.move(src, dst)

# Print folder structure (CSV files listed with size, other files counted)
for name, path in [("JOBS", JOBS_DIR), ("RESUMES", RESUMES_DIR)]:
    print(f"\n=== {name}: {path}")
    for root, dirs, files in os.walk(path):
        depth = root.replace(path, "").count(os.sep)
        csvs = [f for f in files if f.endswith(".csv")]
        others = len(files) - len(csvs)
        print("  " * depth + os.path.basename(root) + "/")
        for f in csvs:
            size_mb = os.path.getsize(os.path.join(root, f)) / 1e6
            print("  " * (depth + 1) + f"{f}  ({size_mb:.1f} MB)")
        if others:
            print("  " * (depth + 1) + f"[+{others} non-CSV files]")

=== JOBS: C:\Users\LENOVO\Documents\Datasets\linkedin-job-postings
linkedin-job-postings/
  postings.csv  (516.8 MB)
  companies/
    companies.csv  (23.2 MB)
    company_industries.csv  (0.8 MB)
    company_specialities.csv  (4.4 MB)
    employee_counts.csv  (1.0 MB)
  jobs/
    benefits.csv  (1.9 MB)
    job_industries.csv  (2.5 MB)
    job_skills.csv  (3.5 MB)
    salaries.csv  (2.3 MB)
  mappings/
    industries.csv  (0.0 MB)
    skills.csv  (0.0 MB)

=== RESUMES: C:\Users\LENOVO\Documents\Datasets\resume-dataset
resume-dataset/
  data/
    data/
      ACCOUNTANT/
        [+118 non-CSV files]
      ADVOCATE/
        [+118 non-CSV files]
      AGRICULTURE/
        [+63 non-CSV files]
      APPAREL/
        [+97 non-CSV files]
      ARTS/
        [+103 non-CSV files]
      AUTOMOBILE/
        [+36 non-CSV files]
      AVIATION/
        [+117 non-CSV files]
      BANKING/
        [+115 non-CSV files]
      BPO/
        [+22 non-CSV files]
      BUSINESS-DEVELOPMENT/
        [+120 non-

## Load and inspect

In [ ]:
import pandas as pd
import os

BASE_DIR = r"C:\Users\LENOVO\Documents\Datasets"
JOBS_DIR = os.path.join(BASE_DIR, "linkedin-job-postings")
RESUMES_DIR = os.path.join(BASE_DIR, "resume-dataset")

postings = pd.read_csv(os.path.join(JOBS_DIR, "postings.csv"))
resumes = pd.read_csv(os.path.join(RESUMES_DIR, "Resume", "Resume.csv"))

for name, df in [("POSTINGS", postings), ("RESUMES", resumes)]:
    print(f"\n=== {name}: {df.shape[0]:,} rows x {df.shape[1]} cols")
    info = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "null_%": (df.isna().mean() * 100).round(1),
        "n_unique": df.nunique(),
    })
    print(info.to_string())

print("\n=== Resume categories")
print(resumes["Category"].value_counts().to_string())

print("\n=== Sample job title + description (first 500 chars)")
row = postings.dropna(subset=["description"]).sample(1, random_state=0).iloc[0]
print(row["title"])
print(row["description"][:500])

=== POSTINGS: 123,849 rows x 31 cols
                              dtype  null_%  n_unique
job_id                        int64     0.0    123849
company_name                    str     1.4     24428
title                           str     0.0     72521
description                     str     0.0    107827
max_salary                  float64    75.9      5321
pay_period                      str    70.9         5
location                        str     0.0      8526
company_id                  float64     1.4     24474
views                       float64     1.4       684
med_salary                  float64    94.9      1417
min_salary                  float64    75.9      4612
formatted_work_type             str     0.0         7
applies                     float64    81.2       274
original_listed_time        float64     0.0     65036
remote_allowed              float64    87.7         1
job_posting_url                 str     0.0    123849
application_url                 str    29.6  

## Category labels for job postings

The resumes have 24 categories. 17 map directly onto LinkedIn's skill tags. The other 7 (chef, fitness, aviation, ...) are industries rather than job functions, so a posting gets one of those labels only if its industry is in an exact list **and** its title contains a matching role keyword.

In [ ]:
job_skills = pd.read_csv(os.path.join(JOBS_DIR, "jobs", "job_skills.csv"))
skills_map = pd.read_csv(os.path.join(JOBS_DIR, "mappings", "skills.csv"))

print("=== job_skills:", job_skills.shape)
print(job_skills.head(3).to_string())

print("\n=== skills mapping:", skills_map.shape)
print(skills_map.to_string())

skills_per_job = job_skills.groupby("job_id").size()
print(f"\nPostings with >=1 skill tag: {skills_per_job.shape[0]:,} / {postings.shape[0]:,}")
print("Skill tags per posting:")
print(skills_per_job.value_counts().sort_index().to_string())

counts = (job_skills.merge(skills_map, on="skill_abr")["skill_name"]
          .value_counts())
print("\n=== Postings per skill category")
print(counts.to_string())

=== job_skills: (213768, 2)
       job_id skill_abr
0  3884428798      MRKT
1  3884428798        PR
2  3884428798       WRT

=== skills mapping: (35, 2)
   skill_abr              skill_name
0        ART            Art/Creative
1       DSGN                  Design
2       ADVR             Advertising
3       PRDM      Product Management
4       DIST            Distribution
5        EDU               Education
6       TRNG                Training
7       PRJM      Project Management
8       CNSL              Consulting
9       PRCH              Purchasing
10      SUPL            Supply Chain
11      ANLS                 Analyst
12      HCPR    Health Care Provider
13      RSCH                Research
14       SCI                 Science
15      GENB        General Business
16      CUST        Customer Service
17      STRA       Strategy/Planning
18       FIN                 Finance
19      OTHR                   Other
20       LGL                   Legal
21       ENG             Engineer

In [ ]:
job_ind = pd.read_csv(os.path.join(JOBS_DIR, "jobs", "job_industries.csv"))
ind_map = pd.read_csv(os.path.join(JOBS_DIR, "mappings", "industries.csv"))

print("=== industries mapping:", ind_map.shape)
print(ind_map.head(3).to_string())

ind_counts = (job_ind.merge(ind_map, on="industry_id")["industry_name"]
              .value_counts())

keywords = {
    "CHEF": ["restaurant", "food", "hospitality"],
    "FITNESS": ["fitness", "wellness", "sport", "recreation"],
    "AGRICULTURE": ["farm", "agricultur", "ranch", "fishery"],
    "AVIATION": ["aviation", "airline", "aerospace"],
    "AUTOMOBILE": ["motor vehicle", "automotive"],
    "APPAREL": ["apparel", "fashion", "textile", "retail apparel"],
    "CONSTRUCTION": ["construction", "civil engineering", "building"],
}

for cat, kws in keywords.items():
    hits = ind_counts[ind_counts.index.str.lower().str.contains("|".join(kws), na=False)]
    print(f"\n=== {cat}: {hits.sum():,} postings")
    print(hits.head(8).to_string())

=== industries mapping: (422, 2)
   industry_id                    industry_name
0            1  Defense and Space Manufacturing
1            3  Computer Hardware Manufacturing
2            4             Software Development

=== CHEF: 6,940 postings
industry_name
Hospitality                        2455
Food and Beverage Services         1880
Food and Beverage Manufacturing    1382
Restaurants                         992
Food and Beverage Retail             93
Food Production                      43
Food & Beverages                     43
Wholesale Food and Beverage          43

=== FITNESS: 4,888 postings
industry_name
Wellness and Fitness Services                          1532
Transportation, Logistics, Supply Chain and Storage    1432
Truck Transportation                                    924
Freight and Package Transportation                      298
Sporting Goods Manufacturing                            223
Spectator Sports                                        172
Health, Well

In [ ]:
import re

# Drop duplicate postings (same description)
postings_d = postings.drop_duplicates(subset=["description"]).copy()

# 17 categories from skill tags
SKILL_TO_CAT = {
    "IT": ["INFORMATION-TECHNOLOGY"], "ENG": ["ENGINEERING"],
    "FIN": ["FINANCE", "BANKING"], "ACCT": ["ACCOUNTANT"],
    "SALE": ["SALES"], "BD": ["BUSINESS-DEVELOPMENT"], "HR": ["HR"],
    "HCPR": ["HEALTHCARE"], "LGL": ["ADVOCATE"], "CNSL": ["CONSULTANT"],
    "PR": ["PUBLIC-RELATIONS"], "EDU": ["TEACHER"], "TRNG": ["TEACHER"],
    "DSGN": ["DESIGNER"], "ART": ["ARTS"],
    "MRKT": ["DIGITAL-MEDIA"], "ADVR": ["DIGITAL-MEDIA"], "WRT": ["DIGITAL-MEDIA"],
    "CUST": ["BPO"],
}

# 7 categories from exact industry list AND title keywords
INDUSTRY_CATS = {
    "CHEF": (["Restaurants", "Food and Beverage Services", "Hospitality"],
             r"\b(chef|cook|culinary|sous|kitchen)\b"),
    "FITNESS": (["Wellness and Fitness Services", "Health, Wellness & Fitness",
                 "Recreational Facilities"],
                r"\b(fitness|trainer|coach|instructor|yoga|pilates|gym)\b"),
    "AGRICULTURE": (["Farming", "Farming, Ranching, Forestry",
                     "Agricultural Chemical Manufacturing", "Wholesale Raw Farm Products"],
                    r"\b(farm\w*|agri\w*|agronom\w*|crop|livestock|horticultur\w*|ranch\w*)\b"),
    "AVIATION": (["Airlines and Aviation", "Aviation and Aerospace Component Manufacturing",
                  "Aviation & Aerospace"],
                 r"\b(pilot|aircraft|aviation|avionics|flight|airframe|a&p)\b"),
    "AUTOMOBILE": (["Motor Vehicle Manufacturing", "Automotive",
                    "Motor Vehicle Parts Manufacturing", "Retail Motor Vehicles"],
                   r"\b(automotive|vehicle|auto|mechanic|technician)\b"),
    "APPAREL": (["Retail Apparel and Fashion", "Apparel & Fashion",
                 "Apparel Manufacturing", "Textile Manufacturing"],
                r"\b(apparel|fashion|merchandis\w*|stylist|garment|textile|sewing)\b"),
    "CONSTRUCTION": (["Construction", "Building Construction", "Civil Engineering"],
                     r"\b(construction|superintendent|foreman|estimator|carpenter|site)\b"),
}
INDUSTRY_CATS = {c: (set(inds), re.compile(p, re.I)) for c, (inds, p) in INDUSTRY_CATS.items()}

skills_by_job = job_skills.groupby("job_id")["skill_abr"].apply(set)
inds_by_job = (job_ind.merge(ind_map, on="industry_id")
               .groupby("job_id")["industry_name"].apply(set))

postings_d["skills"] = postings_d["job_id"].map(skills_by_job)
postings_d["industries"] = postings_d["job_id"].map(inds_by_job)

def assign_labels(row):
    cats = set()
    skills = row["skills"] if isinstance(row["skills"], set) else set()
    inds = row["industries"] if isinstance(row["industries"], set) else set()
    for s in skills:
        cats.update(SKILL_TO_CAT.get(s, []))
    title = str(row["title"])
    for cat, (ind_set, pattern) in INDUSTRY_CATS.items():
        if inds & ind_set and pattern.search(title):
            cats.add(cat)
    return cats

postings_d["labels"] = postings_d.apply(assign_labels, axis=1)

n_labels = postings_d["labels"].apply(len)
print(f"Postings after dedup: {len(postings_d):,}")
print(f"Unlabeled: {(n_labels == 0).sum():,}")
print("\nLabels per posting:")
print(n_labels.value_counts().sort_index().to_string())

print("\n=== Postings per resume category")
print(postings_d["labels"].explode().value_counts().to_string())

Postings after dedup: 107,828
Unlabeled: 31,412

Labels per posting:
labels
0    31412
1    44532
2    22675
3     8842
4      367

=== Postings per resume category
labels
INFORMATION-TECHNOLOGY    23220
SALES                     16574
HEALTHCARE                15048
ENGINEERING               11379
BUSINESS-DEVELOPMENT       9915
FINANCE                    7053
BANKING                    7053
DIGITAL-MEDIA              5055
ACCOUNTANT                 4607
BPO                        2861
HR                         2442
TEACHER                    2145
DESIGNER                   2083
ADVOCATE                   2055
CONSULTANT                 1911
ARTS                       1510
PUBLIC-RELATIONS           1187
CONSTRUCTION                810
AUTOMOBILE                  387
CHEF                        317
AVIATION                     99
APPAREL                      77
FITNESS                      70
AGRICULTURE                  18


## Balanced job corpus

Up to 1,000 postings per category, so large categories (IT, Sales) don't crowd out small ones. AGRICULTURE is dropped (only 18 matching postings).

In [ ]:
import numpy as np

CAP = 1000
DROP = {"AGRICULTURE"}

postings_d["labels"] = postings_d["labels"].apply(lambda s: s - DROP)
labeled = postings_d[postings_d["labels"].apply(len) > 0]

rng = np.random.default_rng(42)
selected = set()
for cat in labeled["labels"].explode().unique():
    ids = labeled.loc[labeled["labels"].apply(lambda s: cat in s), "job_id"].to_numpy()
    take = rng.choice(ids, size=min(CAP, len(ids)), replace=False)
    selected.update(take)

corpus = labeled[labeled["job_id"].isin(selected)][
    ["job_id", "title", "company_name", "location",
     "formatted_experience_level", "description", "labels"]
].copy()
corpus["labels"] = corpus["labels"].apply(sorted)

print(f"Corpus size: {len(corpus):,} postings")
print("\n=== Postings per category in corpus")
print(corpus["labels"].explode().value_counts().to_string())



Corpus size: 17,145 postings

=== Postings per category in corpus
labels
SALES                     3574
INFORMATION-TECHNOLOGY    3298
BANKING                   2568
FINANCE                   2568
DIGITAL-MEDIA             2012
ENGINEERING               1897
BUSINESS-DEVELOPMENT      1882
ACCOUNTANT                1703
DESIGNER                  1543
BPO                       1251
ARTS                      1239
CONSULTANT                1146
TEACHER                   1085
HEALTHCARE                1070
HR                        1040
PUBLIC-RELATIONS          1035
ADVOCATE                  1027
CONSTRUCTION               810
AUTOMOBILE                 387
CHEF                       317
AVIATION                    99
APPAREL                     77
FITNESS                     70


In [ ]:
OUT_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(OUT_DIR, exist_ok=True)
corpus.to_pickle(os.path.join(OUT_DIR, "jobs_corpus.pkl"))
print(f"Saved to {OUT_DIR}")

Saved to C:\Users\LENOVO\Documents\Datasets\processed


## Resume evaluation set

Duplicates and empty resumes removed; 10 resumes per category are set aside for evaluation.

In [ ]:
resumes = pd.read_csv(os.path.join(RESUMES_DIR, "Resume", "Resume.csv"))

resumes = resumes.drop_duplicates(subset=["Resume_str"])
resumes = resumes[resumes["Category"] != "AGRICULTURE"].copy()

# Collapse runs of whitespace left over from PDF extraction
resumes["text"] = resumes["Resume_str"].str.replace(r"\s+", " ", regex=True).str.strip()
resumes = resumes[["ID", "Category", "text"]]

lengths = resumes["text"].str.split().str.len()
print(f"Resumes: {len(resumes):,}  |  categories: {resumes['Category'].nunique()}")
print(f"Words per resume: median {lengths.median():.0f}, "
      f"min {lengths.min()}, max {lengths.max()}")

# Fixed evaluation set: 10 per category
eval_set = resumes.groupby("Category").sample(n=10, random_state=42)
print(f"Eval set: {len(eval_set)} resumes")

print("\n=== Sample resume (first 600 chars)")
print(eval_set.iloc[0]["Category"])
print(eval_set.iloc[0]["text"][:600])

resumes.to_pickle(os.path.join(OUT_DIR, "resumes_all.pkl"))
eval_set.to_pickle(os.path.join(OUT_DIR, "resumes_eval.pkl"))
print(f"\nSaved to {OUT_DIR}")

Resumes: 2,419  |  categories: 23
Words per resume: median 756, min 0, max 5190
Eval set: 230 resumes

=== Sample resume (first 600 chars)
ACCOUNTANT
STAFF ACCOUNTANT Summary Skilled Accountant successful at managing multiple projects and consistently meeting deadlines under pressure. Extensive knowledge of accounting software and processes. Additional ba ckground in payroll, employee management, hiring, training and benefits. Highly detail-oriented and organized. Highlights Financial statement analysis Account reconciliation expert General ledger accounting Effective time management Strong organizational skills Flexible team player Experience 08/2012 to Current Staff Accountant Company Name － City , State Process Receivables (donations) fo

Saved to C:\Users\LENOVO\Documents\Datasets\processed


In [ ]:
resumes = pd.read_pickle(os.path.join(OUT_DIR, "resumes_all.pkl"))

lengths = resumes["text"].str.split().str.len()
print(f"Resumes under 50 words: {(lengths < 50).sum()}")
resumes = resumes[lengths >= 50].copy()

eval_set = resumes.groupby("Category").sample(n=10, random_state=42)
print(f"Resumes kept: {len(resumes):,}  |  Eval set: {len(eval_set)}")

resumes.to_pickle(os.path.join(OUT_DIR, "resumes_all.pkl"))
eval_set.to_pickle(os.path.join(OUT_DIR, "resumes_eval.pkl"))

Resumes under 50 words: 1
Resumes kept: 2,418  |  Eval set: 230
